In [1]:
import torch
from torch.functional import F

class MyMLP(torch.nn.Module):

    def __init__(self):
        super(MyMLP, self).__init__()
        self.fc1 = torch.nn.Linear(28*28, 100)
        self.fc2 = torch.nn.Linear(100, 64)
        self.fc3 = torch.nn.Linear(64, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [2]:
from fluke.data import DataSplitter
from fluke.data.datasets import Datasets
dataset = Datasets.get("mnist", path="./data")
splitter = DataSplitter(dataset=dataset,
                        distribution="iid")


In [3]:
from fluke import DDict
from fluke.utils.log import Log
from fluke.algorithms.fedavg import FedAVG
from fluke.evaluation import ClassificationEval
from fluke import FlukeENV


env = FlukeENV()
env.set_seed(42) # we set a seed for reproducibility
env.set_device("cpu") # we use the CPU for this example
# we set the evaluator to be used by both the server and the clients
env.set_evaluator(ClassificationEval(eval_every=1, n_classes=dataset.num_classes))

client_hp = DDict(
    batch_size=10,
    local_epochs=3,
    loss="CrossEntropyLoss",
    optimizer=DDict(
        lr=0.01,
        momentum=0.9,
        weight_decay=0.0001),
    scheduler=DDict(
        gamma=1,
        step_size=1)
)

hyperparams = DDict(client=client_hp,
                    server=DDict(weighted=True),
                    model=MyMLP())

In [4]:
algorithm = FedAVG(n_clients=10, # 10 clients in the federation
                   data_splitter=splitter,
                   hyper_params=hyperparams)

logger = Log()
algorithm.set_callbacks(logger)

In [5]:
algorithm.run(n_rounds=3, eligible_perc=1)

Output()

╭──────────────────────────────────────────── Round: 1 ────────────────────────────────────────────╮
│ {                                                                                                │
│     'global': {                                                                                  │
│         'accuracy': 0.9454,                                                                      │
│         'macro_precision': 0.94523,                                                              │
│         'macro_recall': 0.94467,                                                                 │
│         'macro_f1': 0.94473,                                                                     │
│         'micro_precision': 0.9454,                                                               │
│         'micro_recall': 0.9454,                                                                  │
│         'micro_f1': 0.9454,                                                                      │
│         'round': 1                                                                               │
│     },                                                                                           │
│     'comm_cost': 1712280                                                                         │
│ }                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

Memory usage: 880.7 MB [5.86 %]

╭──────────────────────────────────────────── Round: 2 ────────────────────────────────────────────╮
│ {                                                                                                │
│     'global': {                                                                                  │
│         'accuracy': 0.9622,                                                                      │
│         'macro_precision': 0.96189,                                                              │
│         'macro_recall': 0.96191,                                                                 │
│         'macro_f1': 0.96185,                                                                     │
│         'micro_precision': 0.9622,                                                               │
│         'micro_recall': 0.9622,                                                                  │
│         'micro_f1': 0.9622,                                                                      │
│         'round': 2                                                                               │
│     },                                                                                           │
│     'comm_cost': 1712280                                                                         │
│ }                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

Memory usage: 884.4 MB [5.88 %]

╭──────────────────────────────────────────── Round: 3 ────────────────────────────────────────────╮
│ {                                                                                                │
│     'global': {                                                                                  │
│         'accuracy': 0.9688,                                                                      │
│         'macro_precision': 0.96853,                                                              │
│         'macro_recall': 0.96856,                                                                 │
│         'macro_f1': 0.96851,                                                                     │
│         'micro_precision': 0.9688,                                                               │
│         'micro_recall': 0.9688,                                                                  │
│         'micro_f1': 0.9688,                                                                      │
│         'round': 3                                                                               │
│     },                                                                                           │
│     'comm_cost': 1712280                                                                         │
│ }                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

Memory usage: 874.7 MB [5.85 %]

Output()

╭────────────────────────────────────── Overall Performance ───────────────────────────────────────╮
│ {                                                                                                │
│     'global': {                                                                                  │
│         'accuracy': 0.9688,                                                                      │
│         'macro_precision': 0.96853,                                                              │
│         'macro_recall': 0.96856,                                                                 │
│         'macro_f1': 0.96851,                                                                     │
│         'micro_precision': 0.9688,                                                               │
│         'micro_recall': 0.9688,                                                                  │
│         'micro_f1': 0.9688,                                                                      │
│         'round': 3                                                                               │
│     },                                                                                           │
│     'comm_costs': 5992980                                                                        │
│ }                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯